# **##  Ingest circuits.csv **file****

1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
- Source File
- Ingestion Timestamp
3. Write to bronze delta table

In [0]:
# Section de création du widget qui controle le batch_id relatif à l'ingestion incrémental
dbutils.widgets.text("p_batch_id", "")          # crée le paramètre, valeur par défaut vide
v_batch_id = dbutils.widgets.get("p_batch_id")  # récupère la valeur dans une variable Python

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

# **Step 1- read the CSV file using the dataframe reader **API****

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header','true')
        # .option('inferSchema','true')
        .option('mode', 'FAILFAST')
        .schema(circuits_schema)
        .load(source_file)
)

In [0]:
display(circuits_df)

# **Step** **2** - **Add** **Metadata** **Columns**

- Source File 
- Ingestion Timestamp

In [0]:
from pyspark.sql import functions as F

circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

# **Step 3 - Write to bronze delta table**

In [0]:
# circuits_final_df = circuits_final_df.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
# (
#     circuits_final_df
#         .write
#         .format('delta')
#         .mode('overwrite')
#         .partitionBy('batch_id')
#         .option('replaceWhere',f"batch_id='{v_batch_id}'") # Permet de overwrite les données relatives à un ceratin batch si jamais !
#         .saveAsTable(table_name)
# )

In [0]:
write_to_bronze(input_df = circuits_final_df, 
                target_table = table_name,
                batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))